# Proyecto 3 – Procesamiento de Lenguaje Natural
## Fine-tuning de DistilBERT para clasificación de emociones en texto

**Autor:** Samuel Osorio Uribe  
**Curso:** Procesamiento de Lenguaje Natural  
**Dataset:** [`dair-ai/emotion`](https://huggingface.co/datasets/dair-ai/emotion) (Hugging Face)  
**Modelo base:** [`distilbert-base-uncased`](https://huggingface.co/distilbert-base-uncased)

> Ejecutar en Google Colab con GPU (Entorno de ejecución → Cambiar tipo → T4 GPU).

**Contenido**
1. Descripción de la base de datos
2. Análisis exploratorio
3. Preprocesamiento (tokenización)
4. Fine-tuning y experimentos de hiperparámetros
5. Mejor modelo y evaluación en test
6. Conclusiones

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

In [ ]:
import random, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, seaborn as sns
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, pipeline)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("GPU disponible:", torch.cuda.is_available())

## 1. Descripción de la base de datos

- **Nombre:** `dair-ai/emotion` (Hugging Face Datasets).
- **Dominio del problema:** análisis de emociones en redes sociales. Son mensajes de Twitter en inglés etiquetados con la emoción principal que expresan.
- **Utilidad:** detectar el estado emocional de un texto sirve para monitoreo de reputación y escucha social, priorizar tickets de servicio al cliente según urgencia emocional, apoyo a sistemas de bienestar y chatbots empáticos.
- **Datos de entrada:** `text`, un mensaje corto en inglés.
- **Variable de salida:** `label`, una de 6 clases: `sadness`, `joy`, `love`, `anger`, `fear`, `surprise`.
- **Tipo de problema:** **clasificación de secuencias (sequence classification) multiclase**.
- **Tamaño:** 16.000 ejemplos de entrenamiento, 2.000 de validación y 2.000 de prueba.
- **Métricas:** *accuracy* y *F1 macro*. Se usa F1 macro porque las clases están desbalanceadas y esta métrica da el mismo peso a cada emoción.
- **Técnica:** fine-tuning de un modelo tipo BERT (DistilBERT) con una cabeza de clasificación sobre el token `[CLS]`.

In [ ]:
dataset = load_dataset("dair-ai/emotion")
print(dataset)
label_names = dataset["train"].features["label"].names
id2label = {i: n for i, n in enumerate(label_names)}
label2id = {n: i for i, n in id2label.items()}
print(id2label)

## 2. Análisis exploratorio de datos

In [ ]:
train_df = dataset["train"].to_pandas()
train_df["label_name"] = train_df["label"].map(id2label)
train_df["n_words"] = train_df["text"].str.split().str.len()
print("Nulos:", train_df.isnull().sum().to_dict())
print("Duplicados en texto:", train_df["text"].duplicated().sum())
train_df.head(10)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
order = train_df["label_name"].value_counts().index
sns.countplot(data=train_df, y="label_name", order=order, ax=ax[0])
ax[0].set_title("Distribución de clases (train)")
sns.histplot(train_df["n_words"], bins=40, ax=ax[1])
ax[1].set_title("Longitud de los textos (palabras)")
plt.tight_layout(); plt.show()
print(train_df["label_name"].value_counts(normalize=True).round(3))
print(train_df["n_words"].describe())

In [ ]:
# Ejemplos por clase para entender el problema
for name in label_names:
    print(f"\n=== {name.upper()} ===")
    for t in train_df[train_df.label_name == name]["text"].sample(3, random_state=SEED):
        print(" -", t)

**Observaciones del EDA:** las clases están desbalanceadas (`joy` y `sadness` dominan; `love` y `surprise` son minoritarias). Los textos son cortos (en su mayoría menos de 30 palabras), por lo que `max_length=128` es más que suficiente. Por el desbalance, la métrica principal será **F1 macro**.

## 3. Preprocesamiento: tokenización

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized = dataset.map(tokenize, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)  # padding dinámico por batch
tokenized

In [ ]:
print(dataset["train"][0]["text"])
print(tokenizer.tokenize(dataset["train"][0]["text"]))

## 4. Fine-tuning y experimentos de hiperparámetros

Se define una función que entrena el modelo con una configuración dada y devuelve las métricas de validación. Se prueban distintos valores de **learning rate**, **batch size**, **épocas** y **weight decay**. Cada corrida usa la misma semilla para que la comparación sea justa. La selección del mejor modelo se hace con **validación**, y el conjunto de **test** se usa una sola vez al final.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro")}

def run_experiment(run_id, lr, batch_size, epochs, weight_decay):
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(label_names), id2label=id2label, label2id=label2id)
    args = TrainingArguments(
        output_dir=f"runs/run_{run_id}",
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=64,
        num_train_epochs=epochs,
        weight_decay=weight_decay,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        seed=SEED,
        report_to="none",
        logging_strategy="epoch",
    )
    trainer = Trainer(model=model, args=args,
                      train_dataset=tokenized["train"], eval_dataset=tokenized["validation"],
                      processing_class=tokenizer, data_collator=data_collator,
                      compute_metrics=compute_metrics)
    trainer.train()
    metrics = trainer.evaluate()
    trainer.save_model(f"runs/run_{run_id}/best")
    return {"run": run_id, "lr": lr, "batch_size": batch_size, "epochs": epochs,
            "weight_decay": weight_decay,
            "val_accuracy": metrics["eval_accuracy"], "val_f1_macro": metrics["eval_f1_macro"]}

In [ ]:
configs = [
    # (lr,   batch, epochs, weight_decay)
    (1e-5, 32, 3, 0.01),
    (2e-5, 32, 3, 0.01),
    (5e-5, 32, 3, 0.01),
    (1e-4, 32, 3, 0.01),   # lr alto: ver si diverge o empeora
    (5e-5, 16, 3, 0.01),   # batch más pequeño
    (5e-5, 32, 5, 0.1),    # más épocas + más regularización
]
results = []
for i, (lr, bs, ep, wd) in enumerate(configs):
    print(f"\n########## Run {i}: lr={lr}, batch={bs}, epochs={ep}, wd={wd} ##########")
    results.append(run_experiment(i, lr, bs, ep, wd))

results_df = pd.DataFrame(results).sort_values("val_f1_macro", ascending=False).reset_index(drop=True)
results_df

In [ ]:
plt.figure(figsize=(8, 4))
labels = [f"run{r.run}\nlr={r.lr}\nbs={r.batch_size}" for r in results_df.itertuples()]
plt.bar(labels, results_df["val_f1_macro"])
plt.ylim(results_df["val_f1_macro"].min() - 0.05, 1.0)
plt.ylabel("F1 macro (validación)"); plt.title("Comparación de hiperparámetros")
plt.tight_layout(); plt.show()

## 5. Mejor modelo y evaluación final en test

In [ ]:
best = results_df.iloc[0]
print("Mejor configuración:\n", best)
best_path = f"runs/run_{int(best.run)}/best"

best_model = AutoModelForSequenceClassification.from_pretrained(best_path)
final_trainer = Trainer(model=best_model, processing_class=tokenizer,
                        data_collator=data_collator, compute_metrics=compute_metrics,
                        args=TrainingArguments(output_dir="runs/final", per_device_eval_batch_size=64,
                                               report_to="none", fp16=torch.cuda.is_available()))
test_out = final_trainer.predict(tokenized["test"])
y_pred = np.argmax(test_out.predictions, axis=-1)
y_true = test_out.label_ids
print("Test accuracy:", round(accuracy_score(y_true, y_pred), 4))
print("Test F1 macro:", round(f1_score(y_true, y_pred, average="macro"), 4))
print(classification_report(y_true, y_pred, target_names=label_names))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_names, yticklabels=label_names)
plt.xlabel("Predicho"); plt.ylabel("Real"); plt.title("Matriz de confusión (test)")
plt.tight_layout(); plt.show()

In [ ]:
# Inferencia con el mejor modelo
clf = pipeline("text-classification", model=best_model, tokenizer=tokenizer,
               device=0 if torch.cuda.is_available() else -1, top_k=None)
ejemplos = ["i am so happy that i passed my exam today",
            "i feel terrified walking alone at night",
            "i can't believe you did that, i am furious",
            "wow i did not expect this gift at all",
            "i miss my family and feel really down"]
for t in ejemplos:
    r = sorted(clf(t)[0], key=lambda x: -x["score"])[0]
    print(f"{t!r} -> {r['label']} ({r['score']:.2f})")

In [ ]:
# Errores del modelo: útil para el análisis
test_df = dataset["test"].to_pandas()
test_df["pred"] = [id2label[i] for i in y_pred]
test_df["real"] = test_df["label"].map(id2label)
test_df[test_df.pred != test_df.real][["text", "real", "pred"]].head(10)

## 6. Conclusiones

> **Completar con tus resultados reales después de ejecutar el notebook.**

- **Mejor modelo:** DistilBERT con lr = ___, batch = ___, épocas = ___, weight decay = ___ → F1 macro en validación = ___, y en test: accuracy = ___, F1 macro = ___.
- **Efecto del learning rate:** (¿qué pasó con 1e-5 vs 5e-5 vs 1e-4? ¿convergió lento, bien o se volvió inestable?).
- **Otros hiperparámetros:** (efecto del batch size, de más épocas y de más weight decay).
- **Análisis de errores:** (qué clases se confunden, por ejemplo `love` con `joy` o `surprise` con `fear`, y por qué: clases minoritarias y emociones semánticamente cercanas).
- **Mejoras posibles:** modelos más grandes (BERT/RoBERTa), balanceo de clases o pesos en la pérdida, aumento de datos, búsqueda automática de hiperparámetros (Optuna).